# Compare Verified vs Exact Global Search Solutions

This notebook compares the **final selected solutions** from two simulation output folders:

- verified-decision run
- exact-global-search run

For each `match_id`, it extracts the decision fields and FE-related statistics from the two output folders, then builds a comparison table with the requested indicators:

- whether the selected PPA type is the same
- whether the selected volume is the same
- whether the selected price is the same
- seller utility difference = **verified − exact global search**

It also keeps the raw solution fields and FE/utility fields from both runs so the result can feed later plotting or diagnostics.

You can switch the target file from:

- `Simulation_Best_Solutions_All_Matches.csv` (default)
- to `Simulation_Best_PPA_Solutions_All_Matches.csv`

by changing `CONFIG["solution_file_name"]`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# Settings
# ============================================================
CONFIG = {
    # Output folders to compare
    "verified_output_dir": "Output files (Risk Neutral, No Mutation, Verified)",
    "exact_output_dir": "Output files (Risk Neutral, No Mutation, Exact Global Search)",

    # Which consolidated file to compare:
    #   "Simulation_Best_Solutions_All_Matches.csv"
    #   "Simulation_Best_PPA_Solutions_All_Matches.csv"
    "solution_file_name": "Simulation_Best_Solutions_All_Matches.csv",

    # Output folder for the comparison results
    "comparison_output_dir": "Output files (Compare Verified vs Exact Global Search)",

    # Optional subset, e.g. [1, 2, 3]
    "selected_match_ids": None,

    # Tolerances for equality checks
    "volume_abs_tol": 1e-9,
    "price_abs_tol": 1e-9,
    "seller_utility_abs_tol": 1e-9,

    # Treat the contract type as the pair (ppa_type, profile_type)
    # This is usually what you want because "Physical AsG" and "Physical Fix"
    # should not be counted as the same contract type.
    "compare_contract_type_using_profile_type": True,

    # Save CSV outputs
    "save_outputs": True,

    # File names for outputs
    "comparison_file_name": "Compared_Solutions_By_Match.csv",
    "summary_file_name": "Compared_Solutions_Summary.csv",
    "mismatch_file_name": "Compared_Solutions_Mismatches_Only.csv",
}


In [ ]:
# ============================================================
# Helpers
# ============================================================
NOTEBOOK_CWD = Path.cwd().resolve()
BUNDLE_ROOT = NOTEBOOK_CWD
WORKSPACE_FOLDER_NAME = "simulation_baseline"


def resolve_simulation_baseline_root() -> Path:
    anchors = [NOTEBOOK_CWD, BUNDLE_ROOT]
    for anchor in anchors:
        anchor = Path(anchor).resolve()
        if anchor.name == WORKSPACE_FOLDER_NAME:
            return anchor
        for parent in anchor.parents:
            if parent.name == WORKSPACE_FOLDER_NAME:
                return parent
        child = anchor / WORKSPACE_FOLDER_NAME
        if child.exists() and child.is_dir():
            return child.resolve()
    return BUNDLE_ROOT


SIMULATION_BASELINE_ROOT = resolve_simulation_baseline_root()


def _search_roots(max_parent_depth: int = 4):
    roots = []
    seen = set()
    for anchor in [SIMULATION_BASELINE_ROOT, BUNDLE_ROOT, NOTEBOOK_CWD, Path("/mnt/data")]:
        p = Path(anchor).expanduser()
        for root in [p, *list(p.parents)[:max_parent_depth]]:
            key = str(root)
            if key not in seen:
                seen.add(key)
                roots.append(root)
    return roots

DEFAULT_COMPARE_COLUMNS = [
    "match_id",
    "scenario_name",
    "ppa_type",
    "profile_type",
    "volume_mw",
    "strike_price_mwh",
    "seller_utility",
    "buyer_utility",
    "seller_mean_exposure",
    "buyer_mean_exposure",
    "seller_exposure_variance",
    "buyer_exposure_variance",
    "expected_seller_revenue",
    "feasible",
    "penalty_rate",
    "buyer_lmp_in_mean",
    "unusual_contracted_volume",
    "best_solution_file",
]

def resolve_folder(folder_like):
    p = Path(folder_like).expanduser()
    if p.exists():
        return p.resolve()

    candidates = []
    for base in _search_roots():
        candidates.append((base / p).resolve())

    for cand in candidates:
        if cand.exists():
            return cand

    tried = [str(p)] + [str(c) for c in candidates]
    raise FileNotFoundError(
        "Could not resolve folder. Tried:\n" + "\n".join(tried)
    )

def anchor_folder(folder_like):
    p = Path(folder_like).expanduser()
    return p.resolve() if p.is_absolute() else (SIMULATION_BASELINE_ROOT / p).resolve()

def resolve_output_csv(folder_like, consolidated_file_name):
    folder = resolve_folder(folder_like)

    direct = folder / consolidated_file_name
    if direct.exists():
        return direct.resolve(), "consolidated"

    # fallback to per-match files
    per_match_root = folder / "Per_Match"
    if per_match_root.exists():
        if consolidated_file_name == "Simulation_Best_Solutions_All_Matches.csv":
            leaf_name = "Best_Solution.csv"
        elif consolidated_file_name == "Simulation_Best_PPA_Solutions_All_Matches.csv":
            leaf_name = "Best_PPA_Solution.csv"
        else:
            leaf_name = None

        if leaf_name is not None:
            leaf_paths = sorted(per_match_root.glob(f"Match_*/{leaf_name}"))
            if leaf_paths:
                return leaf_paths, "per_match"

    raise FileNotFoundError(
        f"Could not find '{consolidated_file_name}' under: {folder}"
    )

def load_solution_table(folder_like, consolidated_file_name):
    resolved, mode = resolve_output_csv(folder_like, consolidated_file_name)

    if mode == "consolidated":
        df = pd.read_csv(resolved)
        source_desc = str(resolved)
    else:
        parts = []
        for p in resolved:
            tmp = pd.read_csv(p)
            if "match_id" not in tmp.columns:
                # attempt to infer from folder name
                try:
                    match_id = int(p.parent.name.split("_")[-1])
                    tmp["match_id"] = match_id
                except Exception:
                    pass
            parts.append(tmp)
        df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        source_desc = f"{len(resolved)} per-match files"

    if "match_id" not in df.columns:
        raise KeyError("The loaded solution table does not contain 'match_id'.")

    df = df.copy()
    df["match_id"] = pd.to_numeric(df["match_id"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["match_id"]).copy()
    df["match_id"] = df["match_id"].astype(int)
    df = df.sort_values("match_id").drop_duplicates(subset=["match_id"], keep="first").reset_index(drop=True)

    return df, source_desc

def canonical_string(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s != "" else None

def same_string(a, b):
    a = canonical_string(a)
    b = canonical_string(b)
    return a == b

def same_number(a, b, abs_tol=1e-9):
    a_isna = pd.isna(a)
    b_isna = pd.isna(b)
    if a_isna and b_isna:
        return True
    if a_isna != b_isna:
        return False
    try:
        return bool(np.isclose(float(a), float(b), atol=abs_tol, rtol=0.0, equal_nan=True))
    except Exception:
        return False

def add_missing_columns(df, required_columns):
    out = df.copy()
    for col in required_columns:
        if col not in out.columns:
            out[col] = np.nan
    return out

def select_and_suffix(df, suffix, compare_columns):
    work = add_missing_columns(df, compare_columns).copy()
    keep = [c for c in compare_columns if c in work.columns]
    work = work[keep].copy()

    rename_map = {c: f"{c}_{suffix}" for c in keep if c != "match_id"}
    work = work.rename(columns=rename_map)
    return work

def build_comparison_table(verified_df, exact_df, config):
    compare_columns = DEFAULT_COMPARE_COLUMNS

    v = select_and_suffix(verified_df, "verified", compare_columns)
    g = select_and_suffix(exact_df, "exact_global", compare_columns)

    merged = v.merge(g, on="match_id", how="outer", indicator=True)
    merged = merged.rename(columns={"_merge": "merge_status"})

    if config["selected_match_ids"] is not None:
        selected = set(int(x) for x in config["selected_match_ids"])
        merged = merged[merged["match_id"].isin(selected)].copy()

    # Requested comparison columns
    if config["compare_contract_type_using_profile_type"]:
        merged["same_ppa_type"] = merged.apply(
            lambda r: same_string(r.get("ppa_type_verified"), r.get("ppa_type_exact_global"))
            and same_string(r.get("profile_type_verified"), r.get("profile_type_exact_global")),
            axis=1,
        )
    else:
        merged["same_ppa_type"] = merged.apply(
            lambda r: same_string(r.get("ppa_type_verified"), r.get("ppa_type_exact_global")),
            axis=1,
        )

    # Helpful extra diagnostic
    merged["same_profile_type"] = merged.apply(
        lambda r: same_string(r.get("profile_type_verified"), r.get("profile_type_exact_global")),
        axis=1,
    )

    merged["same_volume_solution"] = merged.apply(
        lambda r: same_number(
            r.get("volume_mw_verified"),
            r.get("volume_mw_exact_global"),
            abs_tol=config["volume_abs_tol"],
        ),
        axis=1,
    )

    merged["same_price_solution"] = merged.apply(
        lambda r: same_number(
            r.get("strike_price_mwh_verified"),
            r.get("strike_price_mwh_exact_global"),
            abs_tol=config["price_abs_tol"],
        ),
        axis=1,
    )

    merged["seller_utility_diff_verified_minus_global"] = (
        pd.to_numeric(merged.get("seller_utility_verified"), errors="coerce")
        - pd.to_numeric(merged.get("seller_utility_exact_global"), errors="coerce")
    )

    merged["buyer_utility_diff_verified_minus_global"] = (
        pd.to_numeric(merged.get("buyer_utility_verified"), errors="coerce")
        - pd.to_numeric(merged.get("buyer_utility_exact_global"), errors="coerce")
    )

    merged["same_seller_utility_within_tol"] = merged["seller_utility_diff_verified_minus_global"].abs() <= config["seller_utility_abs_tol"]

    merged["same_full_decision"] = (
        merged["same_ppa_type"].fillna(False)
        & merged["same_volume_solution"].fillna(False)
        & merged["same_price_solution"].fillna(False)
    )

    ordered_cols = [
        "match_id",
        "merge_status",

        "ppa_type_verified",
        "profile_type_verified",
        "volume_mw_verified",
        "strike_price_mwh_verified",
        "seller_utility_verified",
        "buyer_utility_verified",
        "seller_mean_exposure_verified",
        "buyer_mean_exposure_verified",
        "seller_exposure_variance_verified",
        "buyer_exposure_variance_verified",
        "feasible_verified",

        "ppa_type_exact_global",
        "profile_type_exact_global",
        "volume_mw_exact_global",
        "strike_price_mwh_exact_global",
        "seller_utility_exact_global",
        "buyer_utility_exact_global",
        "seller_mean_exposure_exact_global",
        "buyer_mean_exposure_exact_global",
        "seller_exposure_variance_exact_global",
        "buyer_exposure_variance_exact_global",
        "feasible_exact_global",

        "same_ppa_type",
        "same_profile_type",
        "same_volume_solution",
        "same_price_solution",
        "same_full_decision",
        "same_seller_utility_within_tol",
        "seller_utility_diff_verified_minus_global",
        "buyer_utility_diff_verified_minus_global",
    ]

    other_cols = [c for c in merged.columns if c not in ordered_cols]
    merged = merged[ordered_cols + other_cols]

    merged = merged.sort_values("match_id").reset_index(drop=True)
    return merged

def build_summary_table(comparison_df):
    both_mask = comparison_df["merge_status"].eq("both")

    summary = {
        "n_rows_total": int(len(comparison_df)),
        "n_rows_in_both": int(both_mask.sum()),
        "n_only_verified": int((comparison_df["merge_status"] == "left_only").sum()),
        "n_only_exact_global": int((comparison_df["merge_status"] == "right_only").sum()),
        "n_same_ppa_type": int((both_mask & comparison_df["same_ppa_type"].fillna(False)).sum()),
        "n_same_profile_type": int((both_mask & comparison_df["same_profile_type"].fillna(False)).sum()),
        "n_same_volume_solution": int((both_mask & comparison_df["same_volume_solution"].fillna(False)).sum()),
        "n_same_price_solution": int((both_mask & comparison_df["same_price_solution"].fillna(False)).sum()),
        "n_same_full_decision": int((both_mask & comparison_df["same_full_decision"].fillna(False)).sum()),
        "mean_abs_seller_utility_diff": float(
            comparison_df.loc[both_mask, "seller_utility_diff_verified_minus_global"].abs().mean()
        ) if both_mask.any() else np.nan,
        "median_abs_seller_utility_diff": float(
            comparison_df.loc[both_mask, "seller_utility_diff_verified_minus_global"].abs().median()
        ) if both_mask.any() else np.nan,
        "max_abs_seller_utility_diff": float(
            comparison_df.loc[both_mask, "seller_utility_diff_verified_minus_global"].abs().max()
        ) if both_mask.any() else np.nan,
    }

    return pd.DataFrame([summary])

def build_mismatch_table(comparison_df):
    both_mask = comparison_df["merge_status"].eq("both")
    mismatch_mask = both_mask & (
        (~comparison_df["same_full_decision"].fillna(False))
        | (~comparison_df["same_seller_utility_within_tol"].fillna(False))
    )
    return comparison_df.loc[mismatch_mask].copy()


In [ ]:
# ============================================================
# Load the two solution tables
# ============================================================
verified_df, verified_source = load_solution_table(
    CONFIG["verified_output_dir"],
    CONFIG["solution_file_name"],
)

exact_df, exact_source = load_solution_table(
    CONFIG["exact_output_dir"],
    CONFIG["solution_file_name"],
)

print("Loaded sources:")
print("  Verified     :", verified_source)
print("  Exact global :", exact_source)
print()
print("Rows loaded:")
print("  Verified     :", len(verified_df))
print("  Exact global :", len(exact_df))


In [ ]:
# ============================================================
# Build the comparison table
# ============================================================
comparison_df = build_comparison_table(
    verified_df=verified_df,
    exact_df=exact_df,
    config=CONFIG,
)

summary_df = build_summary_table(comparison_df)
mismatch_df = build_mismatch_table(comparison_df)

print("Summary:")
display(summary_df)

print("\nFirst rows of the comparison table:")
display(comparison_df.head(20))

print("\nMismatch rows:")
display(mismatch_df.head(20))


In [ ]:
# ============================================================
# Save outputs
# ============================================================
comparison_output_dir = resolve_folder(CONFIG["comparison_output_dir"]) if Path(CONFIG["comparison_output_dir"]).exists() else anchor_folder(CONFIG["comparison_output_dir"])
comparison_output_dir.mkdir(parents=True, exist_ok=True)

comparison_path = comparison_output_dir / CONFIG["comparison_file_name"]
summary_path = comparison_output_dir / CONFIG["summary_file_name"]
mismatch_path = comparison_output_dir / CONFIG["mismatch_file_name"]

if CONFIG["save_outputs"]:
    comparison_df.to_csv(comparison_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    mismatch_df.to_csv(mismatch_path, index=False)

print("Saved:")
print("  Comparison table :", comparison_path)
print("  Summary table    :", summary_path)
print("  Mismatch table   :", mismatch_path)


## Notes

- `same_ppa_type` is defined by default as **both** `ppa_type` and `profile_type` matching.
- `same_volume_solution` treats two missing values as the same.
- `same_price_solution` treats two missing values as the same.
- `seller_utility_diff_verified_minus_global` is exactly:

  `seller_utility_verified - seller_utility_exact_global`

If you want to compare the **best PPA** rather than the final selected decision, change:

```python
CONFIG["solution_file_name"] = "Simulation_Best_PPA_Solutions_All_Matches.csv"
```
